# Model app integration test

This notebook calls `run_model_app()` in `gradientdescentfarmers/app.py` with the editable `USER_PARAMETERS` dictionary. The final output is **one compact DataFrame and one CSV**: each row is one phase of one rotation option. `rotation_id` identifies the alternative, while `rotation_step` and `rotation_phase_count` show its ordered phases. Each row contains water estimates, point SoilGrids nutrient context, SIAP-calibrated yield, estimated harvest price, and gross value. Full PCSE daily and terminal variables are available separately in `result.future_daily_df`.

The example offers a three-phase maize → faba bean → maize option and a two-phase faba bean → maize option. It reads the farm profile and SIAP from GCS, but uses **invented historical and future weather, soil, dates, and areas** for a wiring test. Its yields and values are not validated field forecasts. Supply matching historical weather, measured soil, a documented future-weather scenario, actual crop dates, and cultivated areas for a real scenario. Set Google Application Default Credentials before starting the notebook kernel.

`rotation_options` is an input schedule; this pipeline does not choose crops or carry soil state across its separate season runs. `water_mode` may be `potential`, `rainfed`, or `irrigated`. For irrigation, set a system name and add `irrigation_events` to every historical and planned phase. For potential production, set soil and irrigation settings to `None`.


In [1]:
import sys
from pathlib import Path
import pandas as pd

repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                  if (path / "gradientdescentfarmers" / "model_builder").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Open this notebook within the repository or install the package")
sys.path.insert(0, str(repo_root))

from gradientdescentfarmers.app import PipelineConfig, run_model_app
config_path = repo_root / "resources" / "application.conf"

In [2]:
# Invented weather is only a wiring check. The 2022–2023 portion lets the
# faba-bean simulation overlap the available SIAP calibration years.
def synthetic_weather(start, end):
    return pd.DataFrame({
        "date": pd.date_range(start, end), "tmin_c": 12.0, "tmax_c": 27.0,
        "precip_mm": 2.0, "rh_pct": 65.0, "wind_m_s": 2.0,
        "solar_mj_m2_day": 19.0,
    })

USER_PARAMETERS = {
    "selected_products": ["maize", "fababean"],
    "water_mode": "rainfed",
    "irrigation_system": None,
    "historical_weather": synthetic_weather("2022-01-01", "2025-12-31"),
    "historical_seasons": [
        {"crop": "fababean", "sowing_date": "2022-03-01", "harvest_date": "2022-09-01"},
        {"crop": "fababean", "sowing_date": "2023-03-01", "harvest_date": "2023-09-01"},
        {"crop": "maize", "sowing_date": "2024-05-15", "harvest_date": "2024-12-15"},
        {"crop": "maize", "sowing_date": "2025-05-15", "harvest_date": "2025-12-15"},
    ],
    "rotation_options": [
        {"rotation_id": "maize_faba_maize", "phases": [
            {"crop": "maize", "sowing_date": "2027-05-15",
             "harvest_date": "2027-12-15", "area_ha": 1.0},
            {"crop": "fababean", "sowing_date": "2028-01-15",
             "harvest_date": "2028-06-15", "area_ha": 1.0},
            {"crop": "maize", "sowing_date": "2029-05-15",
             "harvest_date": "2029-12-15", "area_ha": 1.0},
        ]},
        {"rotation_id": "faba_maize", "phases": [
            {"crop": "fababean", "sowing_date": "2027-01-15",
             "harvest_date": "2027-06-15", "area_ha": 1.0},
            {"crop": "maize", "sowing_date": "2028-05-15",
             "harvest_date": "2028-12-15", "area_ha": 1.0},
        ]},
    ],
    "soil_parameters": {
        "SMW": 0.1, "SMFCF": 0.3, "SM0": 0.4, "RDMSOL": 100.0,
        "CRAIRC": 0.06, "SOPE": 10.0, "KSUB": 10.0, "K0": 10.0,
    },
    "initial_available_water_cm": 10.0,
    "future_weather": synthetic_weather("2027-01-01", "2029-12-31"),
    "future_weather_source": "SYNTHETIC_SMOKE",
}

# For irrigated mode, add an event inside every season/phase dictionary, e.g.:
# {"date": "2027-06-01", "amount_mm": 40.0, "efficiency": 0.8}


In [3]:
file_config = PipelineConfig.from_file(config_path)
resolved = {**file_config.model, **USER_PARAMETERS}
print("Products:", resolved["selected_products"])
print("Water mode:", resolved["water_mode"])
for option in resolved["rotation_options"]:
    print("Rotation option:", option["rotation_id"], "phases:", len(option["phases"]))
print("Historical seasons:", len(resolved["historical_seasons"]))
print("Future weather source:", resolved["future_weather_source"])
print("SMOKE TEST: historical/future weather, soil, dates, and areas are invented.")


Products: ['maize', 'fababean']
Water mode: rainfed
Rotation option: maize_faba_maize phases: 3
Rotation option: faba_maize phases: 2
Historical seasons: 4
Future weather source: SYNTHETIC_SMOKE
SMOKE TEST: historical/future weather, soil, dates, and areas are invented.


In [4]:
result = run_model_app(USER_PARAMETERS, config_path=config_path)
final_df = result.final_df
assert len(final_df) == 5
assert final_df.groupby("rotation_id").rotation_step.apply(list).to_dict() == {
    "maize_faba_maize": [1, 2, 3], "faba_maize": [1, 2],
}
assert not any(column.startswith(("summary_", "terminal_")) for column in final_df)
assert "wofost_daily_json" not in final_df
assert "summary_NuptakeTotal" in result.future_daily_df
assert final_df.soil_context_source.eq("SoilGrids_profile").all()
preview = [
    "rotation_id", "rotation_step", "rotation_phase_count", "crop",
    "sowing_date", "harvest_date", "soil_moisture_end_m3_m3",
    "rainfall_total_mm", "transpiration_total_mm",
    "soil_total_n_0_30cm_g_kg", "soil_organic_carbon_0_30cm_g_kg",
    "plant_n_uptake_kg_ha", "calibrated_t_ha", "estimated_tonnes",
    "estimated_price_mxn_t", "estimated_gross_value_mxn",
]
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(final_df[preview])
print("Final DataFrame shape:", final_df.shape)
print("Rows per rotation:", final_df.groupby("rotation_id").size().to_dict())
print("Full WOFOST daily diagnostic shape:", result.future_daily_df.shape)
print("Months through last planned harvest:", result.horizon_months)


,rotation_id,rotation_step,rotation_phase_count,crop,sowing_date,harvest_date,soil_moisture_end_m3_m3,rainfall_total_mm,transpiration_total_mm,soil_total_n_0_30cm_g_kg,soil_organic_carbon_0_30cm_g_kg,plant_n_uptake_kg_ha,calibrated_t_ha,estimated_tonnes,estimated_price_mxn_t,estimated_gross_value_mxn
0,faba_maize,1,2,fababean,2027-01-15,2027-06-15,0.077849,302.0,65.091409,1.784154,13.780163,20.835885,3.668348,3.668348,7280.634532,26707.899931
1,faba_maize,2,2,maize,2028-05-15,2028-12-15,0.087227,428.0,75.224151,1.784154,13.780163,134.230369,6.008909,6.008909,6897.031883,41443.637341
2,maize_faba_maize,1,3,maize,2027-05-15,2027-12-15,0.087178,428.0,75.209962,1.784154,13.780163,134.209064,6.003913,6.003913,6595.891756,39601.157130
3,maize_faba_maize,2,3,fababean,2028-01-15,2028-06-15,0.077470,304.0,65.091409,1.784154,13.780163,20.835885,3.668348,3.668348,7280.634532,26707.899931
4,maize_faba_maize,3,3,maize,2029-05-15,2029-12-15,0.087178,428.0,75.209962,1.784154,13.780163,134.209064,6.003913,6.003913,7074.982717,42477.577352


Final DataFrame shape: (5, 39)
Rows per rotation: {'faba_maize': 2, 'maize_faba_maize': 3}
Full WOFOST daily diagnostic shape: (950, 83)
Months through last planned harvest: 48


In [5]:
SAVE_OUTPUT = True
if SAVE_OUTPUT:
    output_dir = repo_root / "data" / "model_builder"
    name = ("app_synthetic_smoke_final.csv" if resolved["future_weather_source"] == "SYNTHETIC_SMOKE"
            else "app_model_result_final.csv")
    output_path = result.save_final_csv(output_dir / name)
    print("Saved one consolidated CSV:", output_path)


Saved one consolidated CSV: /Users/allisonlara/Desktop/NASA/GradientDescentFarmers/data/model_builder/app_synthetic_smoke_final.csv
